In [ ]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt 
import requests 
import io 
import seaborn as sns 
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance

In [ ]:
NHANES_BASE = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles"

urls = {
    "demo": f"{NHANES_BASE}/DEMO_J.XPT",
    "bmx":  f"{NHANES_BASE}/BMX_J.XPT",
    "dr1":  f"{NHANES_BASE}/DR1TOT_J.XPT",
    "dr2":  f"{NHANES_BASE}/DR2TOT_J.XPT", 
    "paq":  f"{NHANES_BASE}/PAQ_J.XPT",
    "slq":  f"{NHANES_BASE}/SLQ_J.XPT"   
}


In [ ]:
def load_nhanes(name, url):
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers, timeout=30)
    if response.status_code != 200:
        raise Exception(f"Failed to fetch {name}")
    if b"<html" in response.content[:100].lower():
        raise Exception(f"{name} returned HTML - URL is wrong or file moved")
    return pd.read_sas(io.BytesIO(response.content), format='xport', encoding='utf-8')

In [ ]:
demo_data = load_nhanes("demo", urls["demo"])
bmx_data = load_nhanes("bmx", urls["bmx"])
diet1_data = load_nhanes("dr1", urls["dr1"])
diet2_data = load_nhanes("dr2", urls["dr2"])

In [ ]:
demo_data.columns

In [ ]:
demo_data = demo_data[["SEQN", "RIDAGEYR", "RIAGENDR", "RIDRETH3"]]
bmx_data = bmx_data[["SEQN", "BMXWT", "BMXHT", "BMXWAIST", "BMXHIP", "BMXARML"] ]

In [ ]:
metabolic_df = pd.merge(demo_data, bmx_data, on="SEQN", how="inner")

In [ ]:
metabolic_df.isnull().sum()

In [ ]:
metabolic_df = metabolic_df.dropna(subset=["BMXWT"])
metabolic_df["BMXHT"] = metabolic_df.groupby(["RIDAGEYR", "RIAGENDR"])["BMXHT"].transform(lambda x: x.fillna(x.median()))
metabolic_df = metabolic_df.dropna(subset=["BMXHT"])

In [ ]:
corr_cols = ["RIDAGEYR", "RIDAGEYR", "RIDRETH3", "BMXHT", "BMXWAIST", "BMXHIP", "BMXARML"]

In [ ]:
correlation_matrix = metabolic_df[corr_cols].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, cmap="coolwarm")
plt.show()

In [ ]:
target = metabolic_df["BMXWT"]
X = metabolic_df.drop(["SEQN", "BMXWT"], axis=1)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, target, test_size=0.2, random_state=42)

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

In [ ]:
result = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42)
forest_importances = pd.Series(result.importances_mean, index=X_test.columns)
top_features = forest_importances[forest_importances > 0].sort_values(ascending=False)

print("Real Important Features:")
top_features.head(10)

In [ ]:
cols1 = ["SEQN", "DR1TKCAL", "DR1TPROT", "DR1TTFAT", "DR1TCARB", "DR1TSUGR", "DR1TFIBE"]
cols2 = ["SEQN", "DR2TKCAL", "DR2TPROT", "DR2TTFAT", "DR2TCARB", "DR2TSUGR", "DR2TFIBE"]
dietary = diet1_data[cols1].merge(diet2_data[cols2], on="SEQN", how="left")

dietary["calories"] = dietary[["DR1TKCAL", "DR2TKCAL"]].mean(axis=1)
dietary["protein_g"] = dietary[["DR1TPROT", "DR2TPROT"]].mean(axis=1)
dietary["fat_g"]     = dietary[["DR1TTFAT", "DR2TTFAT"]].mean(axis=1)
dietary["carb_g"]    = dietary[["DR1TCARB", "DR2TCARB"]].mean(axis=1)
dietary["sugar_g"]     = dietary[["DR1TSUGR", "DR2TSUGR"]].mean(axis=1)
dietary["fiber_g"]    = dietary[["DR1TFIBE", "DR1TFIBE"]].mean(axis=1)

dietary["calories"] = (dietary["DR1TKCAL"] + dietary["DR2TKCAL"]) / 2
dietary["protein_g"] = (dietary["DR1TPROT"] + dietary["DR2TPROT"]) / 2
dietary["fat_g"] = (dietary["DR1TTFAT"] + dietary["DR2TTFAT"]) / 2
dietary["carb_g"] = (dietary["DR1TCARB"] + dietary["DR2TCARB"]) / 2
dietary["sugar_g"]  = (dietary["DR1TSUGR"] + dietary["DR2TSUGR"]) / 2
dietary["fiber_g"]  = (dietary["DR1TFIBE"] + dietary["DR2TFIBE"]) / 2

In [ ]:
nutrients = ["calories", "protein_g", "fat_g", "carb_g", "sugar_g", "fiber_g"]
dietary = dietary[["SEQN", "calories", "protein_g", "fat_g", "carb_g", "sugar_g", "fiber_g"]]
dietary.dropna(subset=nutrients, inplace=True)

In [ ]:
final_df = metabolic_df.merge(dietary, on="SEQN", how="inner")

In [ ]:
final_df['tdee'] = (10 * final_df['BMXWT']) + (6.25 * final_df['BMXHT']) - (5 * final_df['RIDAGEYR'])
final_df.loc[final_df['RIAGENDR'] == 1, 'tdee'] += 5
final_df.loc[final_df['RIAGENDR'] == 2, 'tdee'] -= 161
final_df['tdee'] *= 1.2

In [ ]:
final_df["sur_def"] = final_df["calories"] - final_df["tdee"]

In [ ]:
len(final_df[final_df['sur_def'] >  0])

In [ ]:
final_df.columns

In [ ]:
target = final_df['BMXWT']
X = final_df[['RIDAGEYR', 'RIAGENDR', 'RIDRETH3', 'BMXHT',
       'BMXWAIST', 'BMXHIP', 'BMXARML', 'calories', 'protein_g', 'fat_g',
       'carb_g', 'sugar_g', 'fiber_g', 'tdee', 'sur_def']]